# Example of XL conlrolled plots

## Imports

In [ ]:
import pandas as pd
from pathlib import Path
import pickle
import matplotlib.dates as mdates
import matplotlib.pyplot as plt
import unicodedata
import re


## def functions

### Importing & exporting dfs and dcts

In [ ]:
#  def write_to_pkl(name):   # name is a string: write_to_pkl("ib_dct")
import pickle
import inspect

def write_to_pkl(name):   # name is a string: write_to_pkl("ib_dct")
    # Access caller frame
    frame = inspect.currentframe().f_back

    # Pull the object from caller's local variables
    if name not in frame.f_locals:
        raise NameError(f"Variable '{name}' does not exist in caller scope")

    obj = frame.f_locals[name]

    # Write to pickle
    filename = f"{name}.pkl"
    with open(filename, "wb") as f:
        pickle.dump(obj, f)

    print(f"{name} written to {filename}")
    return None


In [ ]:
# def read_from_pkl(name):   # name is a string: read_from_pkl("ib_dct")import pickle

import inspect

def read_from_pkl(name):   # name is a string: read_from_pkl("ib_dct")
    filename = f"{name}.pkl"

    # Load the object from pickle
    with open(filename, "rb") as f:
        loaded_obj = pickle.load(f)

    # Assign into caller's local scope
    frame = inspect.currentframe().f_back
    frame.f_locals[name] = loaded_obj

    print(f"{name} updated from {filename}")
    return None



#### XL  ???

### finding the required datcols

In [ ]:
def get_ctrl_row(unq_row_nm, ctrl_dct):                # FINDS THE ROW THAT MATCH PLT_LST_NM AND RETURN THAT ROW OF KEY VALUES
    target = clean_colname(unq_row_nm)                # Makes sure target is clean so it will match one of the cleaned keys

    for row_key, row in ctrl_dct.items():             # cleans all of key0  until it finds a clean one.
        stored = clean_colname(row.get(0))

        if stored == target:
            return [row[k] for k in sorted(row.keys()) if k != 0]

    raise KeyError(f"Control '{unq_row_nm}' not found after cleaning.")


In [ ]:
def single_resolve_col(raw_name, available_cols):  # FINDS AND RETURNS THE NAME THAT MATCHES THE A COL IN THE ALL AVAILABLE COL LIST 
    """
    Clean raw_name and match it to the available column list.
    Returns the actual column name or raises KeyError.
    """
    cleaned = clean_colname(raw_name)

    # Try exact match
    if cleaned in available_cols:
        return cleaned

    # Try case-insensitive match
    for col in available_cols:
        if clean_colname(col).lower() == cleaned.lower():
            return col

    # Try substring match
    for col in available_cols:
        if cleaned.lower() in clean_colname(col).lower():
            return col

    raise KeyError(f"Column '{raw_name}' not found after cleaning.")


In [ ]:
def clean_colname(name):                                 # RETURNS THE CLEANED UP NAME 
    """
    Strip hidden, zero-width, and non-printing characters.
    Normalize Unicode. Remove control chars.
    """
    if not isinstance(name, str):
        name = str(name)

    # Normalize Unicode
    name = unicodedata.normalize("NFKC", name)

    # Remove zero-width characters
    name = re.sub(r"[\u200B-\u200D\uFEFF]", "", name)

    # Remove control characters
    name = re.sub(r"[\x00-\x1F\x7F]", "", name)

    # Strip whitespace (including non-breaking)
    name = name.replace("\xa0", " ").strip()

    return name


In [ ]:
def resolve_col(raw_name, available_cols):
    """
    Clean raw_name and return ALL matching column names.
    Matching rules:
        1. exact match (cleaned)
        2. case-insensitive match
        3. substring match
    Returns a list of matches (may be empty).
    """
    cleaned = clean_colname(raw_name).lower()

    matches = []

    for col in available_cols:
        col_clean = clean_colname(col).lower()

        # exact match
        if col_clean == cleaned:
            matches.append(col)
            continue

        # substring match
        if cleaned in col_clean:
            matches.append(col)

    return matches


In [ ]:
def clean_name(name):
    import unicodedata, re
    
    if not isinstance(name, str):
        name = str(name)
    name = unicodedata.normalize("NFKC", name)
    name = re.sub(r"[\u200B-\u200D\uFEFF]", "", name)
    name = re.sub(r"[\x00-\x1F\x7F]", "", name)
    name = name.replace("\xa0", " ").strip()
    return name


In [ ]:
def lookup_plot(key, id_index, name_index):        # find plot in plot list
    key = clean_name(key)

    # Try row-ID namespace
    if key in id_index:
        return id_index[key]

    # Try key-0 namespace
    if key in name_index:
        return name_index[key]

    raise KeyError(f"'{key}' not found in either namespace.")


### Test Trials of ***def functions***

In [ ]:
read_from_pkl(ib_dct)

In [ ]:
# verify ib_dct

In [ ]:
write_to_pkl(ib_dct)

### creating and editing dfs & dcts

In [ ]:
# Example df
import pandas as pd

data = {
    "row_001": ["icw_vs_ecw", "dtv", "icw", "ecw"],
    "row_002": ["smm_vs_bmi", "dtv", "smm", "bmi"],
}

df_plt_nm = pd.DataFrame.from_dict(data, orient="index",
                                columns=["plt_nm", "x_col", "y1_col", "y2_col"])
df_plt_nm

In [ ]:
def df_to_plt_nm_dct(df):
    out = {}
    for n, (_, row) in enumerate(df.iterrows(), start=1):
        row_id = f"row_{n:03d}"
        out[row_id] = {
            0: row["plt_nm"],
            1: row["x_col"],
            2: row["y1_col"],
            3: row["y2_col"]
        }
    return out                               # output is a "plt_nm_dct" where each is a list of a "plt_nm" and 3 "dat_cols" used to order a plt


##### **Calling Function:** ***plt_nm_dct = df_to_plt_dct(df_plt_nm)***

In [ ]:
df = df_plt_nm
plt_nm_dct = df_to_plt_nm_dct(df)

In [ ]:
# verify plt_nm_dct           # OK

##### ***plt_nm_dct*** = ***df_to_plt_nm_dct(df)***   where ***df dat_cols*** =
1. plt_nm , x_col , y1_col , y2_col
2. row_id & plt_nm_key0 indexes


In [ ]:
def build_dual_index(plt_nm_dct):
    id_index = {}
    name_index = {}

    for row_id, row in plt_nm_dct.items():
        clean_id = clean_name(row_id)
        clean_nm = clean_name(row[0])

        id_index[clean_id] = row
        name_index[clean_nm] = row

    return id_index, name_index


#### Build and using dual‑namespace index

In [ ]:
id_index, name_index = build_dual_index(plt_nm_dct)

In [ ]:
lookup_plot("row_001", id_index, name_index)


In [ ]:
lookup_plot("icw_vs_ecw", id_index, name_index)


## Putting ***ib7797_mrn*** dat_cols to be plotted in a  ***plt_nm***  and that into a ***plt_nm_dct***
  

  out = {}
    for i, row in df.iterrows():
        row_id = f"row_{i+1:03d}"
        out[row_id] = {
            0: row["plt_nm"],
            1: row["x_col"],
            2: row["y1_col"],
            3: row["y2_col"]
        }
    return out
z

### Fill ***plt_nm*** row of dat_cols choosen from the list of available ***dat_cols***
1. First, use  df_77_97_mrn from pickle; to prove & use
2. Later
3. make dct registry with ib_dct & dat_mnl_dct & ....
4. Later, add a filter for mrn, eve, etc
5. Put 

## Workflow using def functions

### Importing the data df_7797_mrn from pickle

In [ ]:

with open("df_77_97_mrn.pkl", "rb") as f:  
    df_77_97_mrn = pickle.load(f)

In [ ]:
df_77_97_mrn.columns = [clean_colname(c) for c in df_77_97_mrn.columns]


In [ ]:
#verify df_77_97_mrn.columns #OK

### working with plt_nm_dct

In [ ]:
# Convert "df to dct"                DataFrame → dict with stable row IDs
df_plt_dct = 
plt_nm_dct = df_to_plt_dct(df_plt_dct)          


## Fill the ***ctrl_dct*** to hold all the info for the plot

### Show ***plt_lst_dct***  =  **plt_nm** , **x_col** , **y1_col** , **y2_col**
1. load ***plt_lst_dct*** from pkl
2. print ***plt_lst_dct*** 

### Add new ***plt_nm***  to ***plt_lst_dct***  ie. add a row  **plt_nm** , **x_col** , **y1_col** , **y2_col**
1. Get ***df_avlbl_col*** cols for plotting 
2. Use  ***resolve_col(raw_name, df_avlbl_col)*** to list candidates ***dat_cols***
3. Create new ***plt_nm*** by pasting in **plt_nm** , **x_col** , **y1_col** , **y2_col**
4. Append to  ***plt_lst_dct*** and load updated  ***plt_lst_dct*** to pkl 

#### Get available cols for plotting 

In [ ]:
df_avlbl_col_tot = df_77_97_mrn.columns    # SHOWS proven AVAILABLE DAT_COLs
df_avlbl_col_tot

#### Create new ***plt_nm*** for the  ***plt_nm_dct***
1. Use ***resolve_col(raw_name, df_avlbl_col)*** to narrow down the search for the desired cols
2. Paste them into a new ***plt_lst_dct*** row ie. **plt_nm** , **x_col** , **y1_col** , **y2_col**
3. Append the new row to ***plt_lst_dct*** with key0 = **plt_nm**

In [ ]:
# narrow down list of candidates with resolve
raw_name = "ecw" 
df_avlbl_col = df_avlbl_col_tot
# Enter a few letters of a name to get a list of available cols that contain that phrase
df_avlbl_col = resolve_col(raw_name, df_avlbl_col)     # REPLACE 
# select and prove the existance
print(df_avlbl_col)

In [ ]:
# narrow down list of candidates with resolve
raw_name = "tbw" 
# Enter a few letters of a name to get a list of available cols that contain that phrase
df_avlbl_col = resolve_col(raw_name, df_avlbl_col)     # REPLACE 
# select and prove the existance
print(df_avlbl_col)

In [ ]:
# narrow down list of candidates with resolve
raw_name = "range" 
# Enter a few letters of a name to get a list of available cols that contain that phrase
df_avlbl_col = resolve_col(raw_name, df_avlbl_col)     # REPLACE 
# select and prove the existance
print(df_avlbl_col)

## Creating ***ctrl_dct*** structure ------- LATER with the ***plt_styl*** values set and placeholders for ***dat_col*** 
1. Test to make work then later
2. Then read for read plt_styl_dct & plt_lst_dct from xl
3. combine ctrl_dct = plt_styl_dct[plt_styl_nm] & plt_lst_dct[plt_lst_nm] from xl

### THIS IS PATTERN FOR "ctrl_dct" WITH STYL values FILLed IN WITH VALUES AND PLACEHOLDERS FOR "dat_cols" for x, y1, y2 in the plot

In [ ]:
ctrl_dct = {                                   # Specifies source of data place holders by df_nm and key_nnm and styl_value are just set
    "row_01": {
        0: "dtv",         # Key0 in ctrl_dct 
        1: "timestamp",     # column name in df
        2: None,            # x-axis has no style attributes
    },

    "row_02": {
        0: "smm_(skeletal_muscle_mass)",
        1: "smm_(skeletal_muscle_mass)",        # dat_col
        2: "red",           # color
        3: .5,             # linewidth
        4: "-",             # linestyle
        5: "o",             # marker
        6: 1              # alpha
    },

    "row_03": {
        0: "ecw/tbw",
        1: "ecw/tbw",
        2: "blue",
        3: .75,
        4: "--",
        5: "s",
        6: 1
      }
}


In [ ]:
ctrl_dct

### OVERWRITE THE "dat_col" OF the prfilled "ctrl_dct"

In [ ]:
#OVERWRITE THE "DAT_COLS" OF "ctrl_dct"

y1= 'weight'
y2 = "ecw/tbw"
ctrl_dct ['row_01'][0]="dtv",                     # name
ctrl_dct ['row_01'][1]="timestamp",                 # dat-col_nm
ctrl_dct ['row_02'][0] = y1,                     # name
ctrl_dct ['row_02'][1] = y1,                     # dat-col_nm
ctrl_dct ['row_03'][0] = y2,                     # name
ctrl_dct ['row_03'][1] = y2                     # dat-col_nm
# verify 
ctrl_dct

### Specify the ***df_nm*** for ***ctrl_dct*** placeholders

#### Specify the ***dat_col-- source ***df_nm***

In [ ]:
df_nm = df_77_97_mrn

##  Use the ***ctrl_dct*** to build the plot command

#### Specify the ***dat_col*** names to be plotted in x-asis, y1, and y2

In [ ]:
ctrl_dct_keys = list(ctrl_dct.keys())
print("ctrl_dct_keys"),ctrl_dct_keys

# NEW version using ***precalculated*** >> ***ctrl_dct***

## ***def function*** definition

In [ ]:
def create_plt_cmmnd(ctll_plt_nm):
    """
    Deterministic dual-axis command generator for the dual engine.
    row_01 = x_axis
    row_02 = y1 (left axis)
    row_03 = y2 (right axis)
    """
  
    cmd_dict = {}

    # -------------------------
    # Extract global X column
    # -------------------------
    x_col = ctll_plt_nm['row_01'][1]
      
    # -------------------------
    # Axis assignment (explicit)
    # -------------------------
    axis_map = {
        'row_02': 'ax',     # left axis
        'row_03': 'ax2'     # right axis
    }

    # -------------------------
    # Build commands for y1 and y2
    # -------------------------
    for row_id in ['row_02', 'row_03']:

        row = ctll_plt_nm[row_id]
        axis = axis_map[row_id]

        # --- Y column ---
        y_raw = row[0]
        y_col = y_raw[0] if isinstance(y_raw, tuple) else y_raw

        # --- Style attributes ---
        color     = row.get(2)
        linewidth = row.get(3)
        linestyle = row.get(4)
        marker    = row.get(5)
        alpha     = row.get(6)

        # --- Build kwargs ---
        kwargs = []
        if color     is not None: kwargs.append(f"color='{color}'")
        if linewidth is not None: kwargs.append(f"linewidth={linewidth}")
        if linestyle is not None: kwargs.append(f"linestyle='{linestyle}'")
        if marker    is not None: kwargs.append(f"marker='{marker}'")
        if alpha     is not None: kwargs.append(f"alpha={alpha}")

        kwargs_str = ", ".join(kwargs)

        # --- Final command ---
        cmd = f"{axis}.plot(df['{x_col}'], df['{y_col}'], {kwargs_str})"
        cmd_dict[row_id] = cmd

    return cmd_dict


## precalculated >> ctrl_dct

In [ ]:
# ctrl_plt_nm          #This is the pattern that holds the style values and dat_col placeholders for a ctrl_plt_nm 
#  it is a dct  with 3 keys [row_01] , [row_02] ,[row_03]
#
ctrl_plt_nm = {'row_01': {0: ('dtv'), 1: ('timestamp'), 2: None},                
 'row_02': {0: ('weight',),
  1: ('weight',),
  2: 'red',
  3: 0.5,
  4: '-',
  5: 'o',
  6: 1},
 'row_03': {0: ('ecw/tbw',),
  1: 'ecw/tbw',
  2: 'blue',
  3: 0.75,
  4: '--',
  5: 's',
  6: 1}}


In [ ]:
# verify ctrl_plt_nm

## Create plot command ***ctrl_plt_nm***

In [ ]:
create_plt_cmmnd(ctrl_plt_nm)    # uses def ***create_plt_cmmnd(ctrl_plt_nm)*** to insert data and style into the plots that can be used by  ***             


In [ ]:
fig, ax = plt.subplots()
ax2 = ax.twinx()

cmds = create_plt_cmmnd(ctrl_plt_nm)

for c in cmds.values():
    exec(c)


In [ ]:
# this is the sme as above don't know I got this.
cmds = [
    "ax.plot(df['timestamp'], df['dtv'])",
    "ax.plot(df['weight'], df['weight'], color='red', linewidth=0.5, linestyle='-', marker='o', alpha=1)",
    "ax.plot(df['ecw/tbw'], df['ecw/tbw'], color='blue', linewidth=0.75, linestyle='--', marker='s', alpha=1)"
]


In [ ]:
df = df_77_97_mrn 
fig, ax = run_plot_commands(cmds, df)


# redo to clean 6pm june 8 see evernote

## Operator‑Grade Plot Command Generator

In [ ]:
def flatten_name(x):
    """Convert ('weight',) → 'weight' and clean."""
    if isinstance(x, tuple):
        x = x[0]
    return clean_name(str(x))

ctrl_plt_nm
def build_plot_command(row, df_name="df", ax_name="ax"):
    """
    Convert a ctrl_plt_nm row into a valid ax.plot() command.
    row = {0: y_col, 1: x_col, 2: color, 3: linewidth, 4: linestyle, 5: marker, 6: alpha}
    """

    # Extract and flatten
    y_col = flatten_name(row[0])
    x_col = flatten_name(row[1])

    color     = row.get(2, None)
    linewidth = row.get(3, None)
    linestyle = row.get(4, None)
    marker    = row.get(5, None)
    alpha     = row.get(6, None)

    # Build argument list
    args = [
        f"{df_name}['{x_col}']",
        f"{df_name}['{y_col}']"
    ]

    kwargs = []
    if color:     kwargs.append(f"color='{color}'")
    if linewidth: kwargs.append(f"linewidth={linewidth}")
    if linestyle: kwargs.append(f"linestyle='{linestyle}'")
    if marker:    kwargs.append(f"marker='{marker}'")
    if alpha:     kwargs.append(f"alpha={alpha}")

    kw = ", ".join(kwargs)

    return f"{ax_name}.plot({', '.join(args)}, {kw})"


In [ ]:
for row_id, row in ctrl_plt_nm.items():
    cmd = build_plot_command(row)
    print(row_id, "→", cmd)


## The Dispatcher Pattern (the correct way)
1. Creates a figure + axis
2. Executes each command in order
3. Applies labels, grid, formatting
4. Returns the figure

In [ ]:
def run_plot_commands(cmd_list, df):              # df is ctrl_plt_nm_dct ?
    import matplotlib.pyplot as plt

    fig, ax = plt.subplots()

    # Execution environment for exec()
    env = {"df": df, "ax": ax}

    for cmd in cmd_list:
        exec(cmd, env)

    ax.grid(True, alpha=0.3)
    return fig, ax


In [ ]:
cmds = [
    "ax.plot(df['timestamp'], df['dtv'])",
    "ax.plot(df['weight'], df['weight'], color='red', linewidth=0.5, linestyle='-', marker='o', alpha=1)",
    "ax.plot(df['ecw/tbw'], df['ecw/tbw'], color='blue', linewidth=0.75, linestyle='--', marker='s', alpha=1)"
]


In [ ]:
df = df_77_97_mrn

In [ ]:
fig, ax = run_plot_commands(cmds, df)


# Later Full plot dispatcher

In [ ]:
def lookup_plot(key, id_index, name_index):
    key = clean_name(key)
    if key in id_index:
        return id_index[key]
    if key in name_index:
        return name_index[key]
    raise KeyError(f"'{key}' not found in registry.")


In [ ]:
def plot_from_registry(key, df, registry, id_index, name_index):
    import matplotlib.pyplot as plt

    # Lookup row
    row = lookup_plot(key, id_index, name_index)

    # Build command
    cmd = build_plot_command(row)

    # Execute
    fig, ax = plt.subplots()
    exec(cmd, {"df": df, "ax": ax})

    ax.set_xlabel(flatten_name(row[1]))
    ax.set_ylabel(flatten_name(row[0]))
    ax.grid(True, alpha=0.3)

    return fig, ax


In [ ]:
id_index, name_index = build_dual_index(row_01 )

fig, ax = plot_from_registry("weight", df, ctrl_plt_nm, id_index, name_index)


In [ ]:
fig, ax = plot_from_registry("row_02", df, ctrl_plt_nm, id_index, name_index)


# One‑Way imports of **Excel Tables** of *dat_cols* ie ***root_tbl***  and **Python Dicts** ie ***root_dct***

In [ ]:
# def xl_dct("data/dat_mnl.xlsm", "dat_mnl")


In [62]:
def xl_dct(xl_path_str, root_name):
    import pandas as pd
    from pathlib import Path
    from openpyxl import load_workbook
    from openpyxl.worksheet.worksheet import Worksheet

    xl_path = Path(xl_path_str)
    tbl_name = f"{root_name}_tbl"

    wb = load_workbook(xl_path, data_only=True)

    sheet_with_table = None
    for sheet in wb.sheetnames:
        ws = wb[sheet]
        if isinstance(ws, Worksheet) and tbl_name in ws.tables:
            sheet_with_table = sheet
            break

    if sheet_with_table is None:
        raise ValueError(f"Table '{tbl_name}' not found in workbook.")

    ws = wb[sheet_with_table]
    tbl = ws.tables[tbl_name]
    ref = tbl.ref  # e.g., "A2:F200"

    # Convert "A2:F200" → "A:F"
    start_cell, end_cell = ref.split(":")
    start_col = ''.join(filter(str.isalpha, start_cell))
    end_col   = ''.join(filter(str.isalpha, end_cell))
    col_range = f"{start_col}:{end_col}"

    df = pd.read_excel(
        xl_path,
        sheet_name=sheet_with_table,
        usecols=col_range,
        header=1,          # <-- THIS IS THE FIX
        engine="openpyxl"
    )

    return {col: df[col] for col in df.columns}


In [63]:
# call the dat_mnl_tbl in WSL copy of XL from table "dat_mnl_tbl" and bring it to jupyterLABs and
#  make dat_mnl_dct loading dat for the col series and keys from the col heads in 2 row

dat_mnl_dct = xl_dct(
    "/home/ratlabs/JL_2/data/dat_mnl/dat_mnl.xlsm",
    "dat_mnl"
)


/home/ratlabs/JL_2/.venv/lib/python3.8/site-packages/openpyxl/worksheet/_reader.py:329: UserWarning: Data Validation extension is not supported and will be removed
  warn(msg)


In [57]:
type(dat_mnl_dct)           # Show it is a dict type

dict

In [58]:
list(dat_mnl_dct.keys())[:10]  # show the first 10 keys 

['col_nms',
 'tst#',
 'dtv',
 'timestamp',
 'Notes',
 'urinePH1_5',
 'bullet coffee',
 'keto_1',
 'Stamina1_5',
 'bd_leg_heat']

In [64]:
# Verify list(dat_mnl_dct.keys())        List all keys   


In [65]:
# verify 
dat_mnl_dct["timestamp","dtv"]  # show datestamp col

KeyError: ('timestamp', 'dtv')